# Model benchmarking: does the model produce realistic full-game score distributions?

The reference papers (LEM in particular) benchmark generative soccer models by
simulating full games and comparing the simulated outcomes to reality. Their
version conditions each simulation on a specific real match (team identity,
lineup) and compares per-match scorelines.

This model has no team/player/match identity in its inputs at all — every
simulation starting from the same initial state (kickoff, empty history) is
drawn from the *same* underlying distribution, there's no way to point it at
"Arsenal vs Chelsea" specifically. So the benchmark here is reframed at the
population level instead of per-game:

- **True target**: the mean and std of home goals / away goals / total goals
  across the 38 val-set games.
- **Simulated**: run the model forward autoregressively N times from the same
  generic kickoff state, extract (home_goals, away_goals) from each simulated
  game, and compute the same mean/std.
- **Compare**: does the simulated distribution's mean and spread match the
  true one?

Everything in this notebook — both the true-score reference numbers and any
seed context, if the generation design ends up using any — comes from the
val set only.

In [1]:
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

pd.set_option("display.max_columns", None)

In [2]:
val = pd.read_csv("data/val_set.csv")

## Reused preprocessing helpers

Copied verbatim from `mlp_embeddings_disjoint_embeddings.ipynb` — same sentinel
convention and normalization the model was trained on, so generated events have
to be encoded the exact same way before going back through the trunk.

In [3]:
def remap_sentinel(values):
    """Undo the -2->106 convention baked into `target` (if present), then
    shift everything into a non-negative space: -2 -> 0, -1 -> 1, real
    category codes 0.. -> 2.. . Works on anything array-like (df columns,
    masks lists, tensors)."""
    values = np.asarray(values, dtype=float)
    values = np.where(values == 106, -2.0, values)
    return (values + 2).astype(np.int64)


def normalize_field(values, mean, std):
    """Standardize real (non-sentinel) values to mean=10, std=1; leaves -1
    (masked) and -2 (no-history) untouched."""
    values = np.asarray(values, dtype=float)
    is_sentinel = (values == -1) | (values == -2)
    return np.where(is_sentinel, values, (values - mean) / std + 10)

In [4]:
continuous_fields = ['start_x', 'start_y', 'time_elapsed', 'duration']
match_state_fields = [
    'period_id', 'minute', 'second',
    'home_goal_count', 'away_goal_count',
    'home_red_count', 'away_red_count',
    'home_yellow_count', 'away_yellow_count',
]
embed_fields = ['is_home', 'event', 'result']
event_cols = ['is_home', 'event', 'start_x', 'start_y', 'time_elapsed', 'duration', 'result']


def with_lags(fields, n_lags=3):
    cols = list(fields)
    for lag in range(1, n_lags + 1):
        cols += [f'{f}-{lag}' for f in fields]
    return cols


continuous_cols = with_lags(continuous_fields) + match_state_fields
embed_cols = {f: [f] + [f'{f}-{lag}' for lag in range(1, 4)] for f in embed_fields}

In [5]:
# Copied from mlp_embeddings_disjoint_embeddings.ipynb -- computed from train
# only, so it's hardcoded here rather than reloading train.csv (this notebook
# only touches the val set). Must match training exactly or normalization
# will be inconsistent with what the model learned.
continuous_stats = {
    'start_x': (53.55268451437807, 24.940933159640434),
    'start_y': (34.141970340665786, 20.15318130576347),
    'time_elapsed': (2.9400464379071893, 6.769398249298148),
    'duration': (1.480916550072, 1.4258817513023883),
    'minute': (44.884499208170936, 27.046770791052666),
    'second': (29.284014497471038, 17.358912964627066),
}

## Load the trained checkpoint

`data/mlp_embeddings_disjoint_checkpoint.pt` — the disjoint-per-field-head
model documented in `MLP_EMBEDDINGS_DISJOINT_EVALUATION.md`.

In [6]:
checkpoint = torch.load("data/mlp_embeddings_disjoint_checkpoint.pt", weights_only=False)

C = {
    "event": checkpoint["C_event"],
    "is_home": checkpoint["C_is_home"],
    "result": checkpoint["C_result"],
}
W1, b1 = checkpoint["W1"], checkpoint["b1"]
W2, b2 = checkpoint["W2"], checkpoint["b2"]
W3, b3 = checkpoint["W3"], checkpoint["b3"]
output_heads = checkpoint["output_heads"]

checkpoint["max_steps"], checkpoint["lossi"][-1]

(1000000, 9.510005950927734)

In [7]:
with open("data/event_masks.json") as f:
    masks = json.load(f)
masks = {col: remap_sentinel(vals).tolist() for col, vals in masks.items()}

with open("data/category_lookup.json") as f:
    lookup = json.load(f)
lookup = {col: {int(k): v for k, v in mapping.items()} for col, mapping in lookup.items()}

output_sizes = {field: len(masks[field]) for field in event_cols}

local_target_lookups = {}
for field in event_cols:
    field_mask = torch.tensor(masks[field])
    n_classes = len(field_mask)
    pos_lookup = torch.full((108,), -1, dtype=torch.long)
    pos_lookup[field_mask] = torch.arange(n_classes)
    local_target_lookups[field] = pos_lookup

# inverse of local_target_lookups: local class index -> shifted global code.
# masks[field][i] is the shifted code assigned local index i above (same
# ordering pos_lookup was built from), so this is a straight lookup, no
# recomputation needed.
local_to_shifted = {field: torch.tensor(masks[field]) for field in event_cols}

## Trunk forward pass

Identical computation to the training loop's forward pass (embed -> concat ->
3x [linear, relu]), generalized to work on a batch of any size (including a
single event) instead of the fixed training/eval batches.

In [8]:
def trunk_forward(X_cont, X_embed):
    """X_cont: (N, len(continuous_cols)) float tensor, already normalized.
    X_embed: dict field -> (N, 4) long tensor of remapped current+lag codes.
    Returns h: (N, n_hidden) post-trunk hidden state."""
    embs = []
    for key in embed_fields:
        emb = C[key][X_embed[key]]
        embs.append(emb.view(emb.shape[0], -1))

    h = torch.cat([X_cont, *embs], dim=1)
    h = torch.relu(h @ W1 + b1)
    h = torch.relu(h @ W2 + b2)
    h = torch.relu(h @ W3 + b3)
    return h

## True score distribution (val set)

In [9]:
true_scores = val.groupby("game_id")[["home_goal_count", "away_goal_count"]].max()
true_scores["total_goals"] = true_scores["home_goal_count"] + true_scores["away_goal_count"]

true_score_stats = true_scores.agg(["mean", "std"]).rename(
    columns={"home_goal_count": "home_goals", "away_goal_count": "away_goals"}
)
true_score_stats

,home_goals,away_goals,total_goals
mean,1.394737,1.342105,2.736842
std,1.103789,1.419484,1.638790


## Generation design

Per-conversation decisions:
- One fixed **start token** (the real kickoff event) seeds every simulation;
  from there the model generates every subsequent event, including `half_end`
  and `full_end` themselves — hitting a generated `full_end` ends the game.
- Row-by-row autoregression: once a row finishes generating, it becomes
  `event-1` (and the old `event-1` -> `event-2`, `event-2` -> `event-3`) for
  the next row — same for all 7 fields, not just `event`.
- Sampling is probabilistic: `softmax(logits) -> torch.multinomial`, so
  repeated sims actually differ.

Two bookkeeping details this required, both grounded in the data rather than
guessed:

- **Goal/card counts** lag by one row in the real data (`home_goal_count`
  reflects state *before* the current row) — confirmed against `stage.csv`:
  `event in {shot, shot_freekick, shot_penalty}` + `result == success` is a
  goal for the row's own team; `result == owngoal` is a goal for the *other*
  team; `yellow_card`/`red_card` results increment that row's own team's
  count (100%/98.5% respectively across the raw data). Applied to the
  *next* row's context, matching the lag.
- **`minute`/`second` are circular within a row**: they're literally
  `cumsum(time_elapsed)` including the current row's own `time_elapsed` —
  verified directly against `val_set.csv` — but `time_elapsed` isn't
  generated until partway through the row (after `is_home`/`event`/
  `start_x`/`start_y`), so those earlier fields necessarily see the clock as
  of the *previous* row (off by a few seconds, typically). The clock is
  refreshed the instant `time_elapsed` is sampled, so `duration`/`result`
  (later in the same row) see the correct value. Also: kickoff's own
  `is_home`/`start_x`/`start_y`/`duration`/`result` are all `-2` (`boundary`)
  and its match-state is `-2` too — but the first *real* generated event
  after it always has match-state `0` (confirmed at `index=1` of every val
  game), which is what the starting state below uses.
- `period_id` flips to 2 the row after a generated `half_end` — the data has
  no separate second-half kickoff token, so this is the only signal for it.

In [10]:
EVENT_KICK_OFF = 0
EVENT_HALF_END = 18
EVENT_FULL_END = 21
RESULT_SUCCESS = 0
RESULT_YELLOW_CARD = 3
RESULT_OWNGOAL = 4
RESULT_RED_CARD = 5
GOAL_EVENTS = {9, 19, 22}  # shot, shot_freekick, shot_penalty


def normalize_value(field, value):
    if field in continuous_fields or field in ("minute", "second"):
        mean, std = continuous_stats[field]
        return float(normalize_field(np.array([value], dtype=float), mean, std)[0])
    return float(value)


def build_features(current, lag1, lag2, lag3, match_state):
    cont_vals = []
    for row in (current, lag1, lag2, lag3):
        for field in continuous_fields:
            cont_vals.append(normalize_value(field, row[field]))

    cont_vals.append(float(match_state["period_id"]))
    cont_vals.append(normalize_value("minute", match_state["minute"]))
    cont_vals.append(normalize_value("second", match_state["second"]))
    for field in [
        "home_goal_count", "away_goal_count",
        "home_red_count", "away_red_count",
        "home_yellow_count", "away_yellow_count",
    ]:
        cont_vals.append(float(match_state[field]))

    X_cont = torch.tensor([cont_vals]).float()

    X_embed = {}
    for field in embed_fields:
        codes = [current[field], lag1[field], lag2[field], lag3[field]]
        X_embed[field] = torch.tensor([remap_sentinel(codes)]).long()

    return X_cont, X_embed


def sample_field(h, field):
    W4, b4 = output_heads[field]
    logits = h @ W4 + b4
    probs = F.softmax(logits, dim=1)
    local_idx = torch.multinomial(probs, num_samples=1).item()
    shifted = local_to_shifted[field][local_idx].item()
    return shifted - 2  # back to raw space (-2 boundary, -1 never a valid class, 0.. real)

In [11]:
def generate_event(lag1, lag2, lag3, match_state):
    """Reveal is_home -> event -> start_x -> start_y -> time_elapsed ->
    duration -> result in order, refeeding each sampled field before
    predicting the next. Returns (current, match_state) -- match_state's
    minute/second are refreshed right after time_elapsed is sampled."""
    current = {f: -1 for f in event_cols}
    match_state = dict(match_state)

    for field in event_cols:
        X_cont, X_embed = build_features(current, lag1, lag2, lag3, match_state)
        with torch.no_grad():
            h = trunk_forward(X_cont, X_embed)
        current[field] = sample_field(h, field)

        if field == "time_elapsed":
            total_seconds = match_state["_total_seconds"] + current["time_elapsed"]
            match_state["_total_seconds"] = total_seconds
            match_state["minute"] = total_seconds // 60
            match_state["second"] = total_seconds % 60

    return current, match_state

In [12]:
def simulate_game(max_events=8000):
    lag1 = {"is_home": -2, "event": EVENT_KICK_OFF, "start_x": -2, "start_y": -2,
            "time_elapsed": 0, "duration": -2, "result": -2}
    lag2 = {f: -2 for f in event_cols}
    lag3 = {f: -2 for f in event_cols}
    match_state = {
        "period_id": 1, "minute": 0, "second": 0, "_total_seconds": 0,
        "home_goal_count": 0, "away_goal_count": 0,
        "home_red_count": 0, "away_red_count": 0,
        "home_yellow_count": 0, "away_yellow_count": 0,
    }

    rows = []
    for i in range(max_events):
        current, match_state = generate_event(lag1, lag2, lag3, match_state)

        row = {"gen_index": i, **current}
        row.update({k: v for k, v in match_state.items() if not k.startswith("_")})
        rows.append(row)

        if current["event"] == EVENT_FULL_END:
            break

        side = "home" if current["is_home"] == 1 else "away" if current["is_home"] == 0 else None
        if side is not None:
            if current["event"] in GOAL_EVENTS and current["result"] == RESULT_SUCCESS:
                match_state[f"{side}_goal_count"] += 1
            elif current["result"] == RESULT_OWNGOAL:
                other = "away" if side == "home" else "home"
                match_state[f"{other}_goal_count"] += 1
            elif current["result"] == RESULT_YELLOW_CARD:
                match_state[f"{side}_yellow_count"] += 1
            elif current["result"] == RESULT_RED_CARD:
                match_state[f"{side}_red_count"] += 1

        if current["event"] == EVENT_HALF_END:
            match_state["period_id"] = 2

        lag3, lag2, lag1 = lag2, lag1, current

    history = pd.DataFrame(rows)
    home_goals = int(history[["home_goal_count", "away_goal_count"]].max()["home_goal_count"])
    away_goals = int(history[["home_goal_count", "away_goal_count"]].max()["away_goal_count"])
    ended_naturally = bool(len(history) and history.iloc[-1]["event"] == EVENT_FULL_END)

    return history, home_goals, away_goals, ended_naturally

In [13]:
# One simulation, timed.
start = time.time()
history, home_goals, away_goals, ended_naturally = simulate_game()
elapsed = time.time() - start

elapsed, len(history), ended_naturally, (home_goals, away_goals)

/var/folders/tv/l4j50kv10gzcv4kzmhysn_fr0000gn/T/ipykernel_83203/971961969.py:39: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:255.)
  X_embed[field] = torch.tensor([remap_sentinel(codes)]).long()


(9.147761106491089, 8000, False, (8, 13))

In [14]:
history

,gen_index,is_home,event,start_x,start_y,time_elapsed,duration,result,period_id,minute,second,home_goal_count,away_goal_count,home_red_count,away_red_count,home_yellow_count,away_yellow_count
0,0,1,1,53,34,0,0,0,1,0,0,0,0,0,0,0,0
1,1,1,1,55,33,0,1,0,1,0,0,0,0,0,0,0,0
2,2,1,2,49,31,2,1,0,1,0,2,0,0,0,0,0,0
3,3,1,1,49,34,1,1,0,1,0,3,0,0,0,0,0,0
4,4,1,2,60,42,1,1,0,1,0,4,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7995,1,1,63,3,3,3,1,1,477,5,8,13,0,0,7,25
7996,7996,0,2,65,3,2,1,0,1,477,7,8,13,0,0,7,25
7997,7997,0,3,63,5,4,1,0,1,477,11,8,13,0,0,7,25
7998,7998,0,2,63,3,1,1,0,1,477,12,8,13,0,0,7,25
